# HomogeneityTest

[All Quick Tutorials](README.md)

Before analyzing a treatment effect, compare the groups on characteristics measured before treatment. A homogeneity check takes the existing assignments as input. For example, a large gap in historical spending can complicate interpretation of a difference in post-treatment spending.

A t-test detects differences in means. The Kolmogorov–Smirnov (KS) test compares the distributions of a numeric metric. For categorical columns, HypEx uses a chi-square test to compare category frequencies.

## Contents

- [What the statistical tests compare](#hom-theory)
  - [Numeric means and distributions](#hom-numeric-tests)
  - [Categorical frequencies and the chi-square test](#hom-chi-square)
- [Compare a historical numeric feature](#hom-numeric)
  - [Simulate baseline imbalance](#hom-simulate-imbalance)
  - [Assign roles and run the test](#hom-assign-numeric-roles)
  - [Read the mean difference and p-values](#hom-read-numeric)
- [Compare category proportions](#hom-category)
  - [Read the chi-square result](#hom-read-category)
- [Check multiple baseline characteristics](#hom-mixed)
  - [Interpret numeric and categorical rows together](#hom-read-mixed)

<a id="hom-theory"></a>

## What the statistical tests compare

<a id="hom-numeric-tests"></a>

### Numeric means and distributions

A t-test checks whether the means of a numeric feature are equal. The KS test compares distributions using the maximum gap between their empirical cumulative distribution functions.

$$
D_{n,m}=\sup_x|\widehat F_C(x)-\widehat F_T(x)|.
$$

<a id="hom-chi-square"></a>

### Categorical frequencies and the chi-square test

For categorical data, compare observed counts $O_{gk}$ with expected counts $E_{gk}$ under equal category proportions across groups. Pearson's chi-square statistic is

$$
\chi^2=\sum_{g,k}\frac{(O_{gk}-E_{gk})^2}{E_{gk}},\qquad
E_{gk}=\frac{n_{g\cdot} n_{\cdot k}}{n}.
$$

SciPy applies Yates's correction by default when there is one degree of freedom. Under the test's assumptions, a small p-value indicates that the data are unlikely under the null hypothesis. [NIST two-sample KS test](https://itl.nist.gov/div898/software/dataplot/refman1/auxillar/ks2samp.htm) · [SciPy contingency table test](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html)

<a id="hom-numeric"></a>

## Compare a historical numeric feature

<a id="hom-simulate-imbalance"></a>

### Simulate baseline imbalance

Generate data in which assignment depends on the historical feature `X_lag_1`.

The generator defaults to `dependent_division=True`. The example spells it out to show that assignment depends on the baseline feature.

In [1]:
from hypex import HomogeneityTest
from hypex.dataset import Dataset, TargetRole, TreatmentRole
from hypex.utils.tutorial_data_creation import gen_control_variates_df

frame = gen_control_variates_df(
    data_size=1200, dependent_division=True, random_state=42
)
frame[["Treatment", "X_lag_1"]].head(3)

,Treatment,X_lag_1
0,0,1.899464
1,1,0.472709
2,1,9.207838


The first rows show the group label and historical feature value. The test uses all 1,200 rows.

<a id="hom-assign-numeric-roles"></a>

### Assign roles and run the test

The simulated assignment depends on `X_lag_1`. Assign `TreatmentRole` to the existing group column and `TargetRole` to the historical feature you want to check.

Call the parameterless `HomogeneityTest()` constructor, then pass the dataset with its roles to `execute(data)`.

In [2]:
data = Dataset(
    roles={"Treatment": TreatmentRole(), "X_lag_1": TargetRole()},
    data=frame,
)
result = HomogeneityTest().execute(data)
result.summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,X_lag_1,1,1.271988,3.459913,2.187925,172.008319,1.929678,2.446173,NOT OK,7.012739e-56,NOT OK,2.235408e-44


<a id="hom-read-numeric"></a>

### Read the mean difference and p-values

The historical feature averages about 2.19 units higher in the treatment group. Both tests give p-values below 0.05. Consider the size of the difference and its statistical uncertainty when reading the report.

<a id="hom-category"></a>

## Compare category proportions

Convert the historical feature into `low` and `high` categories. Assign `TargetRole(str)` to compare category frequencies in control and treatment.

In [3]:
frame["segment"] = (frame["X_lag_1"] >= 2.5).map({False: "low", True: "high"})
categorical = Dataset(
    roles={"Treatment": TreatmentRole(), "segment": TargetRole(str)},
    data=frame,
)
HomogeneityTest().execute(categorical).summary

,feature,group,Chi2Test pass,Chi2Test p-value
0,segment,1,NOT OK,5.734396e-21


<a id="hom-read-category"></a>

### Read the chi-square result

The `segment` row reports `Chi2Test p-value`. A small value indicates different proportions. Compare the group proportions directly to assess the size of the imbalance.

To create candidate groups, see [AATest](AATest.ipynb).

<a id="hom-mixed"></a>

## Check multiple baseline characteristics

You can check several baseline characteristics for the same assignments. Give the numeric `X_lag_1` and categorical `segment` columns `TargetRole`, and HypEx applies the appropriate tests to each.

Use `feature` to find the rows in the full `summary`. For the numeric row, read `difference`. For the categorical row, read `Chi2Test p-value`.

In [4]:
mixed_data = Dataset(
    roles={
        "Treatment": TreatmentRole(),
        "X_lag_1": TargetRole(),
        "segment": TargetRole(str),
    },
    data=frame,
)
HomogeneityTest().execute(mixed_data).summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value,KSTest pass,KSTest p-value,Chi2Test pass,Chi2Test p-value
0,X_lag_1,1,1.271988,3.459913,2.187925,172.008319,1.929678,2.446173,NOT OK,7.012739e-56,NOT OK,2.235408e-44,NaN,NaN
1,segment,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOT OK,5.734396e-21


<a id="hom-read-mixed"></a>

### Interpret numeric and categorical rows together

One row refers to the numeric feature, and the other to category proportions. `difference` shows a difference in numeric means. Since `segment` is derived from `X_lag_1`, the two checks are related.

See [homogeneity checks on pandas and Spark](SparkBackend.ipynb#spark-homogeneity).